## Silver to Gold Job

Constrói a camada Gold do IMDM (Índice de Maturidade Digital Municipal) — Etapa 3 conforme `docs/arquitetura/gold_dimensional_model.md`. **O radar não está mais no código:** dimensões, itens, regras e pontuação vêm do radar da edição, definido pelo especialista em `config/radar/radar_imdm.xlsx` e carregado pelo `pipeline/load_radar_config` nas tabelas `config.radar_*`.

- 1. Configuração, guarda de pré-requisitos (Ajuste 1, radar da edição carregado) e criação do schema `semantic`
- 2. `dim_municipio` — municípios com porte populacional derivado
- 3. Radar IMDM da edição — lido de `config.radar_variavel` (dimensões, itens, âncoras e regras)
- 4. `dim_indicador` — variáveis com significados, linhagem e regras de pontuação
- 5. `fact_resposta` — despivotização atômica (municípios × indicadores com dados)
- 6. `fact_resposta` — pontuação no nível do item, genérica por regra, com os pontos de `config.radar_pontuacao`
- 7. `fact_resposta` — montagem final com denominador efetivo e gravação
- 8. `fact_score_dimensao` (município × dimensão) e `fact_imdm_municipio` (IMDM = média ponderada das dimensões do ano que entram no IMDM; nível pelas faixas de `config.radar_nivel`)
- 9. Controles automatizados de qualidade (Seção 6 da especificação)

> **Pré-requisito crítico (Ajuste 1):** a `silver.munic_clean_data` deve preservar os 6 estados textuais originais da MUNIC (`Sim`, `Não`, `-`, `Não sabe informar`, `Não informou`, `Recusa`). O Step 1 bloqueia a execução caso contrário.

> **Pré-requisito do radar:** o `pipeline/load_radar_config` precisa ter rodado para a edição (linha `radar` no ledger e linhas da edição em `config.radar_*`). Cada edição tem o seu radar — 2019 e 2024 podem ter dimensões diferentes, e nada no código fixa quantas são.

In [0]:
%run ../environment/env_variables

In [0]:
%run ../environment/env_functions

## Step 1 - Configuração, pré-requisitos e schema semântico

- Define as tabelas de origem (silver + diagnósticos `dq_*` da Etapa 2) e as tabelas gold de destino
- Valida que as 360 colunas de indicadores estão em `StringType` com os 6 estados originais preservados (Ajuste 1) — a execução é bloqueada se o cast destrutivo ainda estiver vigente
- Cria o schema `semantic` (Ajuste 3) que receberá as views do notebook `gold_to_semantic`

In [ ]:
from pyspark.sql import functions as F

# Tabelas de origem (silver + diagnósticos da Etapa 2)
silver_data_table  = f"{catalog_name}.{silver_schema}.munic_clean_data"
silver_dict_table  = f"{catalog_name}.{silver_schema}.munic_clean_dict"
dq_profile_table   = f"{catalog_name}.{gold_schema}.dq_column_profile"
dq_skiplogic_table = f"{catalog_name}.{gold_schema}.dq_skiplogic_candidates"

# Radar da edição (definido na planilha do especialista; carregado pelo load_radar_config)
config_schema          = "config"
radar_dimensao_table   = f"{catalog_name}.{config_schema}.radar_dimensao"
radar_variavel_table   = f"{catalog_name}.{config_schema}.radar_variavel"
radar_pontuacao_table  = f"{catalog_name}.{config_schema}.radar_pontuacao"
radar_nivel_table      = f"{catalog_name}.{config_schema}.radar_nivel"

# Tabelas gold de destino
dim_municipio_table       = f"{catalog_name}.{gold_schema}.dim_municipio"
dim_indicador_table       = f"{catalog_name}.{gold_schema}.dim_indicador"
fact_resposta_table       = f"{catalog_name}.{gold_schema}.fact_resposta"
fact_score_dimensao_table = f"{catalog_name}.{gold_schema}.fact_score_dimensao"
fact_imdm_table           = f"{catalog_name}.{gold_schema}.fact_imdm_municipio"

# Ajuste 3: schema semântico para as views do gold_to_semantic
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.semantic")

# Carrega a edição corrente da silver
silver_df = spark.table(silver_data_table).filter(F.col("ano_edicao") == reference_year)

# Fronteira silver→gold contra o CONTRATO do ledger (pipeline_manifest — seção 5.3)
manifest_silver = contrato_da_edicao(etapa="silver", ano_edicao=reference_year)
# Radar da edição: o load_radar_config valida a planilha contra esta silver e declara a linha 'radar'
manifest_radar = contrato_da_edicao(etapa="radar", ano_edicao=reference_year)

# Colunas da EDIÇÃO: no schema união da silver, colunas que só existem em outra
# edição chegam 100% nulas para a corrente — a fact_resposta só pode despivotizar
# as colunas com dados (as nulas virariam milhões de células 'não informou' fantasma).
todas_indicadoras = [c for c in silver_df.columns if c.startswith(("mtic", "mgov"))]
indicator_cols = colunas_com_dados(silver_df, todas_indicadoras)   # env_functions (modulos/ledger)

# Check: shape com dados da edição == indicadores declarados pela silver
check_igual(len(indicator_cols), manifest_silver["indicadores"], "silver: indicadores com dados",
              contexto=f"detalhes da silver: {manifest_silver['detalhes']}")
# Check: nº de municípios == linhas declaradas pela silver
check_igual(silver_df.count(), manifest_silver["linhas"], "silver: municípios")
non_string_cols = [c for c in indicator_cols if silver_df.schema[c].dataType.simpleString() != "string"]
if non_string_cols:
    raise Exception(
        "Pré-requisito 'Ajuste 1' NÃO aplicado: colunas de indicadores não são StringType "
        f"(cast destrutivo ainda vigente, ex.: {non_string_cols[:8]}...). "
        "Reexecute o bronze_to_silver preservando os 6 estados originais da MUNIC "
        "('Sim', 'Não', '-', 'Não sabe informar', 'Não informou', 'Recusa') antes de rodar o silver_to_gold."
    )

resumo_validacoes()   # qualquer FAIL interrompe o run aqui (fail-fast), antes de construir a gold

print(f"OK: {len(indicator_cols)} indicadores em StringType (6 estados preservados)")
print(f"Edição {reference_year} | municípios: {silver_df.count()} | radar sha256 {manifest_radar['sha256_xlsx'][:12]}…")

## Step 2 - `dim_municipio` (1 linha por município × edição)

- Origem: colunas cadastrais e descritivas da `silver.munic_clean_data`
- Deriva o `porte_municipio` a partir de `populacao_total` (Seção 3.1): Pequeno I (até 20 mil), Pequeno II (20–50 mil), Médio (50–100 mil), Grande I (100–500 mil) e Grande II (mais de 500 mil)

In [0]:
# porte_municipio derivado da população (Seção 3.1 da especificação)
porte_municipio_expr = (
    F.when(F.col("populacao_total") <= 20000, "Pequeno I")
     .when(F.col("populacao_total") <= 50000, "Pequeno II")
     .when(F.col("populacao_total") <= 100000, "Médio")
     .when(F.col("populacao_total") <= 500000, "Grande I")
     .otherwise("Grande II")
)

dim_municipio = silver_df.select(
    "cod_ibge", "ano_edicao", "nome", "uf_sigla", "regiao",
    "populacao_total", "faixa_populacao",
    porte_municipio_expr.alias("porte_municipio"),
)

upsert_delta_table(
    df_source=dim_municipio,
    table_name=dim_municipio_table,
    merge_keys=["cod_ibge", "ano_edicao"],
    ano_edicao=reference_year,
    description="Etapa 3 (silver_to_gold): dim_municipio com porte populacional derivado",
)
display(dim_municipio.limit(10))

## Step 3 - Radar IMDM da edição (lido de `config.radar_*`)

- O radar é **definido por edição** pelo especialista (`config/radar/radar_imdm.xlsx`) e carregado pelo `load_radar_config`, que já o validou contra esta silver — aqui não há lista fixa de itens nem de dimensões
- Cada item vale no máximo 1 ponto; cada dimensão = pontos obtidos / pontos possíveis × 100
- Formato interno (o mesmo de antes): `(dimensao, variável-âncora do item, regra, variáveis-membro)`; `dimensao` = `dimensao_codigo` da edição (ex.: `servicos_digitais_ao_cidadao`)
- Em itens multi-variável, os pontos ficam apenas na variável-âncora (1ª variável do item na planilha); as demais preservam o estado original para auditoria (Seção 3.2.1)

**Regras de pontuação:** `SIM_POSITIVO` (Sim = 1), `CATEGORICO_ESCALONADO` (pontos por resposta, ou por variável marcada 'Sim' no formato de publicação), `ESCADA_MAIOR_NIVEL` (maior nível marcado), `GRUPO_PROPORCIONAL` (fração de Sim do grupo) e `NAO_PONTUA` (fora do índice)

In [ ]:
# Radar da edição: uma linha por item × variável (config.radar_variavel)
SIM_POS    = "SIM_POSITIVO"
CAT_ESC    = "CATEGORICO_ESCALONADO"
ESCADA     = "ESCADA_MAIOR_NIVEL"
GRUPO      = "GRUPO_PROPORCIONAL"
NAO_PONTUA = "NAO_PONTUA"
REGRAS_CONHECIDAS = {SIM_POS, CAT_ESC, ESCADA, GRUPO}
EXCECAO_LAI = "NAO_SE_APLICA_SEM_PEDIDO_LAI"

radar_var = (
    spark.table(radar_variavel_table)
    .filter(F.col("edicao") == reference_year)
    .orderBy("item", "ordem")
    .toPandas()
)
radar_dims = (
    spark.table(radar_dimensao_table)
    .filter(F.col("edicao") == reference_year)
    .orderBy("ordem")
    .toPandas()
)
radar_pts = spark.table(radar_pontuacao_table).filter(F.col("edicao") == reference_year).toPandas()
if radar_var.empty or radar_dims.empty:
    raise RuntimeError(
        f"Radar da edição {reference_year} não encontrado em {catalog_name}.{config_schema}. "
        "Rode o pipeline/load_radar_config para esta edição (o especialista precisa ter "
        "preenchido as linhas do ano em config/radar/radar_imdm.xlsx)."
    )

# Mesma estrutura de antes: (dimensao, âncora, regra, membros) — agora vinda da planilha
imdm_items = [
    (g["dimensao_codigo"].iloc[0], g["cod_variable"].iloc[0], g["regra"].iloc[0], g["cod_variable"].tolist())
    for _, g in radar_var.groupby("item", sort=False)
]
item_nome     = {g["cod_variable"].iloc[0]: item for item, g in radar_var.groupby("item", sort=False)}
item_excecao  = {g["cod_variable"].iloc[0]: g["excecao"].iloc[0] for _, g in radar_var.groupby("item", sort=False)}
ancoras_excecao_lai = [a for a, e in item_excecao.items() if e == EXCECAO_LAI]

# Dimensões do ano: código, nome, ordem, peso e se entram no IMDM
dims_edicao = {
    r.dimensao_codigo: {"nome": r.dimensao, "ordem": int(r.ordem), "peso": float(r.peso),
                        "entra": str(r.entra_no_imdm).lower() == "sim"}
    for r in radar_dims.assign(dimensao_codigo=radar_dims["dimensao_codigo"]).itertuples()
}

# Sanidade (o load_radar_config já validou; aqui garante que gold e radar não divergiram)
from collections import Counter
regras_desconhecidas = sorted({r for _, _, r, _ in imdm_items} - REGRAS_CONHECIDAS)
assert not regras_desconhecidas, f"Regras sem implementação no pipeline: {regras_desconhecidas}"
dims_sem_cadastro = sorted({d for d, _, _, _ in imdm_items} - set(dims_edicao))
assert not dims_sem_cadastro, f"Itens apontam para dimensões fora de radar_dimensao: {dims_sem_cadastro}"
all_members = [v for _, _, _, members in imdm_items for v in members]
assert len(all_members) == len(set(all_members)), "Uma variável pertence a mais de um item do radar"
missing_members = [v for v in all_members if v not in indicator_cols]
assert not missing_members, (
    f"Variáveis do radar {reference_year} sem dados nesta silver: {missing_members} — "
    "corrija a planilha do radar (o load_radar_config deveria ter barrado)."
)
dim_counts = Counter(d for d, _, _, _ in imdm_items)
check_igual(len(imdm_items), int(manifest_radar["linhas"]), f"radar {reference_year}: itens = contrato do load_radar_config")

# cod_variable -> (dimensão, âncora do item, regra, é âncora?)
imdm_scoring_map = {
    v: (d, anchor, rule, v == anchor)
    for d, anchor, rule, members in imdm_items
    for v in members
}
resumo_validacoes()
print(f"Radar {reference_year}: {len(imdm_items)} itens em {len(dims_edicao)} dimensões | "
      f"{len(imdm_scoring_map)} variáveis pontuadas | {len(indicator_cols) - len(imdm_scoring_map)} NAO_PONTUA")
for cod, d in sorted(dims_edicao.items(), key=lambda kv: kv[1]["ordem"]):
    print(f"  {d['ordem']}. {d['nome']} ({cod}) — {dim_counts.get(cod, 0)} itens | peso {d['peso']:g} | entra no IMDM: {d['entra']}")

## Step 4 - `dim_indicador` (1 linha por variável × edição; 360 linhas)

- Base do JOIN: as 360 colunas reais da silver via `dq_column_profile` (Ajuste 2) — descarta os códigos fantasmas do dicionário do IBGE e herda os 7 significados reconstruídos na Etapa 2 + a correção de `mtic30`
- `grupo_pergunta`: pai confirmado na `dq_skiplogic_candidates`; caso contrário, radical extraído por regex
- `dimensao_imdm`, `item_imdm`, `peso_indicador` e `regra_pontuacao` vêm do radar do Step 3
- Papel duplo (COBIT APO14.02): glossário de negócios ativo + tabela de parametrização — mudar peso ou regra de um indicador exige apenas atualizar esta tabela

In [0]:
# Guard de pré-requisito (Etapa 2 -> Etapa 3): a edição corrente precisa ter sido
# perfilada e PERSISTIDA pelo bronze_quality_profiling (persistir_tabelas='sim').
# As tabelas dq_* acumulam todas as edições (2019, 2024, ...) — a gold só pode usar
# as linhas da edição corrente; sem elas, aborta em vez de usar evidência de outro ano.
def _exigir_dq_da_edicao(nome_tabela):
    if not spark.catalog.tableExists(nome_tabela):
        raise RuntimeError(
            f"Pré-requisito ausente: {nome_tabela} não existe. Rode o notebook "
            f"pipeline/bronze_quality_profiling com persistir_tabelas='sim' "
            f"para a edição {reference_year} antes de construir a gold."
        )
    n = spark.table(nome_tabela).filter(F.col("ano_edicao") == reference_year).count()
    if n == 0:
        raise RuntimeError(
            f"Pré-requisito ausente: {nome_tabela} não tem linhas da edição {reference_year}. "
            f"Rode o notebook pipeline/bronze_quality_profiling com "
            f"persistir_tabelas='sim' para essa edição."
        )
    print(f"Pré-requisito OK: {nome_tabela} — {n} linhas da edição {reference_year}")

_exigir_dq_da_edicao(dq_profile_table)
_exigir_dq_da_edicao(dq_skiplogic_table)

# Base: as 360 colunas reais da silver (via dq_column_profile — Ajuste 2),
# restritas à edição corrente — sem o filtro por ano_edicao, cada variável viria
# de TODAS as edições perfiladas e o MERGE do upsert falharia por chave duplicada
profile_df = (
    spark.table(dq_profile_table)
    .filter(F.col("ano_edicao") == reference_year)
    .filter(F.col("coluna").isin(indicator_cols))
    .select(
        F.col("coluna").alias("cod_variable"),
        F.col("significado"),
        F.col("origem_significado"),
        F.col("tipo_detectado"),
    )
)

# Correção documental do grupo (Ajuste 2, Seção 1.1): o dicionário oficial do IBGE
# traz "educação" em mtic30 por erro de texto; o correto é a área de COMUNICAÇÃO
# (consistente com o bloco mtic31/mtic331 de comunicação social).
CORRECAO_MTIC30 = "Estrutura organizacional para a área de comunicação - existência"
profile_df = profile_df.withColumns(
    {
        "significado": F.when(F.col("cod_variable") == "mtic30", F.lit(CORRECAO_MTIC30)).otherwise(F.col("significado")),
        "origem_significado": F.when(F.col("cod_variable") == "mtic30", F.lit("corrigido pelo grupo")).otherwise(F.col("origem_significado")),
    }
)

# Fallback de significado a partir do dicionário silver
dict_df = (
    spark.table(silver_dict_table)
    .filter(F.col("ano_edicao") == reference_year)
    .select("cod_variable", "meaning")
)

# Pais confirmados por salto lógico (Etapa 2)
# - o bronze_quality_profiling grava o pai em "coluna_pai" (a doc antiga citava "coluna_pai_candidata"):
#   aceita os dois nomes para não quebrar conforme a versão da tabela dq_ no workspace
# - a tabela guarda o melhor pai de CADA filha, inclusive os de relação "parcial": só
#   veredito_dados = 'sim' vira grupo_pergunta; as demais caem no radical por regex
skiplogic_raw = (
    spark.table(dq_skiplogic_table)
    .filter(F.col("ano_edicao") == reference_year)   # só a edição corrente
)
pai_col = next((c for c in ("coluna_pai", "coluna_pai_candidata") if c in skiplogic_raw.columns), None)
assert pai_col, f"{dq_skiplogic_table}: coluna do pai não encontrada (colunas: {skiplogic_raw.columns})"
veredito_col = next((c for c in ("veredito_dados", "relacao_confirmada") if c in skiplogic_raw.columns), None)
if veredito_col:
    skiplogic_raw = skiplogic_raw.filter(F.col(veredito_col) == "sim")
skiplogic_df = skiplogic_raw.select(
    F.col("coluna_filha").alias("cod_variable"),
    F.col(pai_col).alias("pai_confirmado"),
)
print(f"Pais de salto lógico confirmados: {skiplogic_df.count()} (coluna do pai: {pai_col})")

# Parametrização do radar (Step 3) como lookup
radar_lookup = spark.createDataFrame(
    [
        (cod, d, anchor, rule, 1.0 if is_anchor else None)
        for cod, (d, anchor, rule, is_anchor) in imdm_scoring_map.items()
    ],
    schema="cod_variable string, dimensao_imdm string, item_imdm string, "
          "regra_pontuacao string, peso_indicador double",
)

dim_indicador = (
    profile_df
    .join(dict_df, on="cod_variable", how="left")
    .join(skiplogic_df, on="cod_variable", how="left")
    .join(radar_lookup, on="cod_variable", how="left")
    .select(
        "cod_variable",
        F.lit(reference_year).cast("int").alias("ano_edicao"),
        F.coalesce("significado", "meaning").alias("meaning"),
        "origem_significado",
        F.when(F.col("cod_variable").startswith("mtic"), "TIC").otherwise("GOV").alias("tema"),
        F.col("tipo_detectado").alias("tipo_resposta"),
        # grupo_pergunta: pai confirmado na dq_skiplogic_candidates; senão radical por regex
        # (Seção 3.2.1: ex. mtic12a1 -> mtic12). Questão-base sem sufixo não tem pai -> NULL.
        F.coalesce(
            F.when(F.col("pai_confirmado") != "", F.col("pai_confirmado")),
            F.when(
                F.regexp_extract("cod_variable", r"^(mtic|mgov)\d{2}", 0) != F.col("cod_variable"),
                F.regexp_extract("cod_variable", r"^(mtic|mgov)\d{2}", 0),
            ),
        ).alias("grupo_pergunta"),
        F.coalesce("dimensao_imdm", F.lit("Nao_Pontua")).alias("dimensao_imdm"),
        F.col("item_imdm"),          # NULL quando Nao_Pontua
        F.col("peso_indicador"),      # 1.0 na variável-âncora do item, NULL nas demais
        F.coalesce("regra_pontuacao", F.lit("NAO_PONTUA")).alias("regra_pontuacao"),
    )
)

upsert_delta_table(
    df_source=dim_indicador,
    table_name=dim_indicador_table,
    merge_keys=["cod_variable", "ano_edicao"],
    ano_edicao=reference_year,
    description="Etapa 3 (silver_to_gold): dim_indicador com regras de pontuação do radar IMDM",
)
display(dim_indicador.limit(10))

## Step 5 - `fact_resposta`: despivotização atômica (5.570 × 360 = 2.005.200 linhas)

- `stack` converte as 360 colunas de indicadores em linhas (município × variável)
- `classe_preenchimento`: as 5 classes mutuamente exclusivas validadas na Etapa 2 — `Valida`, `Nao_Aplicavel_Skip` (salto lógico `-`), `Nao_Sabe`, `Nao_Informou` e `Recusa`
- `resposta_normalizada`: `'1'`/`'0'` para os estados binários; categorias e texto livre preservados; NULL em `Recusa`

In [0]:
# Despivota as 360 colunas de indicadores em linhas atômicas (município × variável)
stack_pairs = ", ".join(f"'{c}', `{c}`" for c in indicator_cols)
resposta_long = silver_df.select(
    "cod_ibge", "ano_edicao",
    F.expr(f"stack({len(indicator_cols)}, {stack_pairs}) as (cod_variable, resposta_original)"),
).withColumn("resposta_original", F.col("resposta_original").cast("string"))

# Rótulos de não-resposta comparados sem acento, sem caixa e sem espaços nas pontas, para aceitar
# variações de redação entre edições da MUNIC (ex.: "Não sabe informar", "Não sabe", "Não soube informar").
_SEM_ACENTO = ("ÃãÁáÀàÂâÉéÊêÍíÓóÔôÕõÚúÇç", "AaAaAaAaEeEeIiOoOoOoUuCc")
resposta_chave = F.lower(F.trim(F.translate(F.col("resposta_original"), *_SEM_ACENTO)))
RE_NAO_SABE = r"^nao (sabe|soube)( informar)?$"

# classe_preenchimento: as 5 classes mutuamente exclusivas validadas na Etapa 2
classe_preenchimento_expr = (
    F.when(resposta_chave == "recusa", "Recusa")
     .when(resposta_chave.rlike(RE_NAO_SABE), "Nao_Sabe")
     .when(resposta_chave == "nao informou", "Nao_Informou")
     .when(F.trim(F.col("resposta_original")) == "-", "Nao_Aplicavel_Skip")
     .when(F.col("resposta_original").isNull() | (F.trim(F.col("resposta_original")) == ""), "Nao_Informou")
     .otherwise("Valida")
)

# resposta_normalizada: '1'/'0' para estados binários; categoria/texto preservados; NULL em Recusa
# (derivada da classe, para tratar igual qualquer redação dos rótulos de não-resposta)
resposta_normalizada_expr = (
    F.when(F.col("resposta_original") == "Sim", "1")
     .when(F.col("classe_preenchimento") == "Recusa", F.lit(None).cast("string"))
     .when((F.col("resposta_original") == "Não")
           | F.col("classe_preenchimento").isin("Nao_Sabe", "Nao_Informou", "Nao_Aplicavel_Skip"), "0")
     .otherwise(F.col("resposta_original"))
)

fact_resposta_base = (
    resposta_long
    .withColumn("classe_preenchimento", classe_preenchimento_expr)
    .withColumn("resposta_normalizada", resposta_normalizada_expr)
    .join(
        dim_indicador.select(
            "cod_variable", "ano_edicao", "dimensao_imdm", "item_imdm",
            "regra_pontuacao", "peso_indicador",
        ),
        on=["cod_variable", "ano_edicao"],
        how="left",
    )
)
print(f"Linhas atômicas: {fact_resposta_base.count()} (esperado: 2.005.200)")

# Aviso: resposta classificada como 'Valida' mas com cara de não-resposta (rótulo novo numa edição)
suspeitas = (fact_resposta_base.where((F.col("classe_preenchimento") == "Valida")
                                      & resposta_chave.rlike(r"^nao (sab|soub|inform)|^recus"))
             .groupBy("resposta_original").count().collect())
for r in suspeitas:
    print(f"AVISO: resposta '{r['resposta_original']}' ({r['count']} linhas) ficou como 'Valida' — "
          "se for um rótulo de não-resposta, inclua-o na classificação acima")
if not suspeitas:
    print("OK: nenhum rótulo de não-resposta fora da classificação")

## Step 6 - `fact_resposta`: pontuação no nível do item

Genérico por regra — nenhum código de variável escrito aqui; os pontos vêm de `config.radar_pontuacao` da edição. Os pontos são atribuídos à variável-âncora:

- **`CATEGORICO_ESCALONADO` com 1 variável** (ex.: `mtic10`, `mgov09`, `mtic29`, `mgov05`): pontos por resposta exata; resposta fora do mapa pontua 0 e gera AVISO
- **`CATEGORICO_ESCALONADO` com várias variáveis** (formato de publicação) e **`ESCADA_MAIOR_NIVEL`** (canais): vale a maior pontuação entre as variáveis marcadas 'Sim'
- **`GRUPO_PROPORCIONAL`**: fração de 'Sim' entre as variáveis do grupo
- **Exceção `NAO_SE_APLICA_SEM_PEDIDO_LAI`** (item do `mgov05`): 'Não publica' + `mgov04` = 'Não' → sai do denominador

Semântica do denominador (Seções 3.3.2 e 3.3.3): `-`, 'Não sabe informar' e 'Não informou' pontuam 0 com peso ativo; `Recusa` (na âncora) anula pontos e peso.

In [ ]:
import re
import unicodedata
from functools import reduce

# Itens pontuados no nível de item: âncoras com regra != SIM_POSITIVO
anchors_item = {
    anchor: (dim, regra, members)
    for dim, anchor, regra, members in imdm_items
    if regra != SIM_POS
}
ESTADOS_NAO_PONTUAVEIS = {"Recusa", "Não sabe informar", "Não informou", "-"}
def pontos_do_item(anchor):
    """{(variável, resposta): pontos} do item na edição (config.radar_pontuacao)."""
    g = radar_pts[radar_pts["item"] == item_nome[anchor]]
    return {(r.cod_variable, r.resposta): float(r.pontos) for r in g.itertuples()}

item_scores_parts = []
avisos_step6 = []
for anchor, (dim, regra, members) in anchors_item.items():
    is_recusa = F.col(anchor) == "Recusa"
    excecao = (anchor in ancoras_excecao_lai)
    nao_se_aplica = F.lit(False)
    if excecao:
        if "mgov04" in indicator_cols:
            nao_se_aplica = (F.col(anchor) == "Não publica") & (F.col("mgov04") == "Não")
        else:
            avisos_step6.append(f"mgov04 sem dados na edição {reference_year} — exceção do 'não publica' ({anchor}) não avaliável")

    if regra == GRUPO:
        sim_count = reduce(lambda a, b: a + b,
                           [F.when(F.col(c) == "Sim", F.lit(1.0)).otherwise(F.lit(0.0)) for c in members])
        pontos = sim_count / F.lit(float(len(members)))
    else:
        mapa = pontos_do_item(anchor)
        if len(members) == 1:
            # Escalonado de 1 variável: pontos por resposta exata
            valores = [r[0] for r in silver_df.select(anchor).where(F.col(anchor).isNotNull()).distinct().collect()]
            nao_cobertos = sorted(v for v in set(valores) - {r for (_, r) in mapa} - ESTADOS_NAO_PONTUAVEIS
                                  if not re.match(RE_NAO_SABE, unicodedata.normalize("NFKD", v).encode("ascii", "ignore").decode().strip().lower()))
            if nao_cobertos:
                avisos_step6.append(f"{anchor}: valores fora do mapa do radar — pontuam 0.0 até mapeamento: {nao_cobertos}")
            pontos = F.coalesce(*[F.when(F.col(anchor) == r, F.lit(p)) for (_, r), p in mapa.items()], F.lit(0.0))
        else:
            # Várias variáveis (formato de publicação / escada): maior pontuação entre as marcadas 'Sim'
            por_var = [F.when(F.col(v) == "Sim", F.lit(p)).otherwise(F.lit(0.0)) for (v, _), p in mapa.items()]
            pontos = F.greatest(*por_var) if len(por_var) > 1 else por_var[0]

    item_scores_parts.append(
        silver_df.select(
            "cod_ibge", "ano_edicao",
            F.lit(anchor).alias("cod_variable"),
            F.when(is_recusa | nao_se_aplica, F.lit(None).cast("double")).otherwise(pontos).alias("pontos_item"),
            F.when(is_recusa | nao_se_aplica, F.lit(None).cast("double")).otherwise(F.lit(1.0)).alias("peso_item"),
        )
    )

for a in avisos_step6:
    print(f"AVISO: {a}")

if item_scores_parts:
    item_scores = reduce(lambda a, b: a.unionByName(b), item_scores_parts)
else:   # radar só com SIM_POSITIVO
    item_scores = spark.createDataFrame([], "cod_ibge string, ano_edicao int, cod_variable string, pontos_item double, peso_item double")
n_ancoras_item = item_scores.select("cod_variable").distinct().count()
assert n_ancoras_item == len(anchors_item), (
    f"Âncoras pontuadas no nível de item: {n_ancoras_item} (esperado pelo radar da edição: {len(anchors_item)})"
)
print(f"Âncoras pontuadas no nível de item: {n_ancoras_item} (esperado pelo radar da edição: {len(anchors_item)})")

## Step 7 - `fact_resposta`: montagem final e gravação

- `pontos_obtidos` e `peso_aplicado` ficam apenas na variável-âncora do item; variáveis `NAO_PONTUA`, membros não-âncora e células de `Recusa` recebem NULL nas duas colunas
- No 'não se aplica' legítimo (item com a exceção `NAO_SE_APLICA_SEM_PEDIDO_LAI`), a `resposta_normalizada` e o `pontos_obtidos` ficam NULL (Seção 3.3.3)
- Gravação via `upsert_delta_table` com chave composta `(cod_ibge, cod_variable, ano_edicao)`

In [ ]:
# Pontos: SIM_POSITIVO é resolvido na própria linha; regras de item usam os pontos do Step 6
pontos_obtidos_expr = (
    F.when(F.col("regra_pontuacao") == "NAO_PONTUA", F.lit(None).cast("double"))
     .when(F.col("classe_preenchimento") == "Recusa", F.lit(None).cast("double"))
     .when(F.col("peso_indicador").isNull(), F.lit(None).cast("double"))   # membro não-âncora
     .when(
         F.col("regra_pontuacao") == SIM_POS,
         F.when(F.col("resposta_original") == "Sim", F.col("peso_indicador"))   # 1.0 × peso
         .otherwise(F.lit(0.0)),                                                 # 'Não'/'-'/'Não sabe'/'Não informou'
     )
     .otherwise(
         # 'não se aplica' legítimo (exceção da LAI) -> pontos NULL, sai do denominador (Seção 3.3.3)
         F.when(
             F.col("cod_variable").isin(ancoras_excecao_lai)
             & (F.col("peso_item").isNull())
             & (F.col("classe_preenchimento") == "Valida"),
             F.lit(None).cast("double"),
         ).otherwise(F.coalesce("pontos_item", F.lit(0.0)))   # demais regras de item (âncora)
     )
)

# Peso: ativo no denominador, exceto NAO_PONTUA, Recusa, não-âncoras e a exceção de mgov05
peso_aplicado_expr = (
    F.when(F.col("regra_pontuacao") == "NAO_PONTUA", F.lit(None).cast("double"))
     .when(F.col("classe_preenchimento") == "Recusa", F.lit(None).cast("double"))
     .when(F.col("peso_indicador").isNull(), F.lit(None).cast("double"))
     .when(F.col("regra_pontuacao") == SIM_POS, F.col("peso_indicador"))
     .otherwise(F.col("peso_item"))
)

fact_resposta = (
    fact_resposta_base
    .join(item_scores, on=["cod_ibge", "ano_edicao", "cod_variable"], how="left")
    .withColumn(
        # 'não se aplica' legítimo (exceção da LAI) -> resposta_normalizada NULL (Seção 3.3.3)
        "resposta_normalizada",
        F.when(
            F.col("cod_variable").isin(ancoras_excecao_lai)
            & (F.col("peso_item").isNull())
            & (F.col("classe_preenchimento") == "Valida"),
            F.lit(None).cast("string"),
        ).otherwise(F.col("resposta_normalizada")),
    )
    .select(
        "cod_ibge", "ano_edicao", "cod_variable",   # ano_edicao em 2º: ordem que o upsert_delta_table mantém
        "resposta_original", "resposta_normalizada", "classe_preenchimento",
        pontos_obtidos_expr.alias("pontos_obtidos"),
        peso_aplicado_expr.alias("peso_aplicado"),
    )
)

upsert_delta_table(
    df_source=fact_resposta,
    table_name=fact_resposta_table,
    merge_keys=["cod_ibge", "cod_variable", "ano_edicao"],
    ano_edicao=reference_year,
    description="Etapa 3 (silver_to_gold): fact_resposta atômica com pontuação por item e denominador efetivo",
)
display(fact_resposta.limit(10))

## Step 8 - `fact_score_dimensao` e `fact_imdm_municipio`

- **`fact_score_dimensao`** (1 linha por município × dimensão da edição): `Score_d = (Σ pontos_obtidos / Σ peso_aplicado) × 100` — longa, porque o nº de dimensões varia por edição
- **`fact_imdm_municipio`** (1 linha por município × edição): `IMDM = Σ (peso_d × Score_d) / Σ peso_d` sobre as dimensões do ano com `entra_no_imdm = sim` (pesos da planilha; pesos iguais = média simples)
- NULL: `Recusa` (município com recusa formal) ou `Sem_Item_Valido` (alguma dimensão do IMDM sem item válido)
- `nivel_maturidade_mmd`: faixas da edição em `config.radar_nivel` (rótulo `"<nível>-<nome>"`)

> Esquema novo da `fact_imdm_municipio` (sem as 5 colunas fixas `score_d*`): rode uma vez `DROP TABLE munic_catalog.gold.fact_imdm_municipio` antes do primeiro run desta versão — o upsert recria a tabela.

In [ ]:
# Pontos e pesos por município × dimensão (dimensões da edição vindas do radar)
dim_lookup = dim_indicador.select("cod_variable", "ano_edicao", "dimensao_imdm")
flag_recusa = (
    fact_resposta.groupBy("cod_ibge", "ano_edicao")
    .agg(F.max(F.when(F.col("classe_preenchimento") == "Recusa", 1).otherwise(0)).alias("flg_recusa"))
)

dims_pdf = spark.createDataFrame(
    [(cod, d["nome"], d["ordem"], d["peso"], d["entra"]) for cod, d in dims_edicao.items()],
    "dimensao_codigo string, dimensao_nome string, dimensao_ordem int, dimensao_peso double, entra_no_imdm boolean",
)

somas = (
    fact_resposta.join(dim_lookup, on=["cod_variable", "ano_edicao"], how="left")
    .where(F.col("dimensao_imdm") != "Nao_Pontua")
    .groupBy("cod_ibge", "ano_edicao", F.col("dimensao_imdm").alias("dimensao_codigo"))
    .agg(F.sum("pontos_obtidos").alias("pontos_obtidos"), F.sum("peso_aplicado").alias("pontos_possiveis"))
)
# Toda combinação município × dimensão do ano (inclusive sem soma), para o NULL ficar explícito
fact_score_dimensao = (
    silver_df.select("cod_ibge", "ano_edicao").crossJoin(dims_pdf)
    .join(somas, on=["cod_ibge", "ano_edicao", "dimensao_codigo"], how="left")
    .withColumn(
        "score",
        F.when(F.col("pontos_possiveis") > 0,
               F.round(F.col("pontos_obtidos") / F.col("pontos_possiveis") * 100, 2))
         .otherwise(F.lit(None).cast("double")),
    )
    .select("cod_ibge", "ano_edicao", "dimensao_codigo", "dimensao_nome", "dimensao_ordem",
            "dimensao_peso", "entra_no_imdm", "pontos_obtidos", "pontos_possiveis", "score")
)

upsert_delta_table(
    df_source=fact_score_dimensao,
    table_name=fact_score_dimensao_table,
    merge_keys=["cod_ibge", "ano_edicao", "dimensao_codigo"],
    ano_edicao=reference_year,
    description="Etapa 3 (silver_to_gold): score por município × dimensão do radar da edição",
)

# IMDM = média ponderada das dimensões do ano que entram no índice
n_dims_imdm = sum(1 for d in dims_edicao.values() if d["entra"])
imdm_agg = (
    fact_score_dimensao.where(F.col("entra_no_imdm"))
    .groupBy("cod_ibge", "ano_edicao")
    .agg(
        F.sum(F.col("dimensao_peso") * F.col("score")).alias("soma_ponderada"),
        F.sum(F.when(F.col("score").isNotNull(), F.col("dimensao_peso"))).alias("soma_pesos"),
        F.count("score").alias("n_dimensoes_pontuadas"),
    )
)

# Nível de maturidade: faixas da edição (o último nível inclui o máximo, 100)
niveis = (spark.table(radar_nivel_table).filter(F.col("edicao") == reference_year)
          .orderBy("minimo").collect())
assert niveis, f"Faixas de maturidade da edição {reference_year} ausentes em {radar_nivel_table}"
nivel_expr = F.lit(None).cast("string")
for i, n in enumerate(reversed(niveis)):
    rotulo = f"{int(n['nivel'])}-{n['nome']}"
    cond = (F.col("score_global_imdm") >= n["minimo"]) & (
        (F.col("score_global_imdm") <= n["maximo"]) if i == 0 else (F.col("score_global_imdm") < n["maximo"])
    )
    nivel_expr = F.when(cond, F.lit(rotulo)).otherwise(nivel_expr)

fact_imdm_municipio = (
    flag_recusa.join(imdm_agg, on=["cod_ibge", "ano_edicao"], how="left")
    .withColumn(
        "score_global_imdm",
        F.when((F.col("flg_recusa") == 0) & (F.col("n_dimensoes_pontuadas") == n_dims_imdm),
               F.round(F.col("soma_ponderada") / F.col("soma_pesos"), 2))
         .otherwise(F.lit(None).cast("double")),
    )
    .withColumn(
        "motivo_imdm_nulo",
        F.when(F.col("flg_recusa") == 1, F.lit("Recusa"))
         .when(F.col("score_global_imdm").isNull(), F.lit("Sem_Item_Valido"))
         .otherwise(F.lit(None).cast("string")),
    )
    .withColumn("nivel_maturidade_mmd", F.when(F.col("score_global_imdm").isNotNull(), nivel_expr))
    .withColumn("n_dimensoes_imdm", F.lit(n_dims_imdm))
    .withColumn("radar_sha256", F.lit(manifest_radar["sha256_xlsx"]))
    .select(
        "cod_ibge", "ano_edicao", "motivo_imdm_nulo", "score_global_imdm", "nivel_maturidade_mmd",
        "n_dimensoes_imdm", F.coalesce("n_dimensoes_pontuadas", F.lit(0)).alias("n_dimensoes_pontuadas"),
        "radar_sha256",
    )
)

upsert_delta_table(
    df_source=fact_imdm_municipio,
    table_name=fact_imdm_table,
    merge_keys=["cod_ibge", "ano_edicao"],
    ano_edicao=reference_year,
    description="Etapa 3 (silver_to_gold): IMDM global (média ponderada das dimensões do radar da edição) e estágio MMD",
)
display(fact_imdm_municipio.limit(10))

## Step 9 - Controles automatizados de qualidade (Seção 6)

Todos os esperados vêm da edição corrente e do **radar da edição** (nenhum nº de itens ou de dimensões escrito no código):

1. **Volume de indicadores:** nº de colunas com dados da edição (contrato da silver), temas TIC/GOV, âncoras do radar e distribuição por regra
2. **Conservação:** `municípios × indicadores` linhas na fact, Recusa preservada contra o contrato da silver e classes fechando o total; peso máximo por dimensão = nº de itens da dimensão no radar
3. **IMDM:** `fact_score_dimensao` = municípios × dimensões do ano; IMDM NULL = recusas + municípios sem item válido em alguma dimensão do índice; IMDM recalculado de forma independente pela média ponderada bate com o gravado; scores em [0, 100]
4. **Integridade referencial:** 0 registros órfãos nas FKs fato → dimensões
5. **Proteção ética:** `mgov15`/`mgov16` fora do índice — verificado quando presentes na edição

Qualquer FAIL interrompe o pipeline com `AssertionError`.

In [ ]:
dim_municipio_gold  = spark.table(dim_municipio_table).filter(F.col("ano_edicao") == reference_year)
dim_indicador_gold = spark.table(dim_indicador_table).filter(F.col("ano_edicao") == reference_year)
fact_resposta_gold = spark.table(fact_resposta_table).filter(F.col("ano_edicao") == reference_year)
fact_score_gold    = spark.table(fact_score_dimensao_table).filter(F.col("ano_edicao") == reference_year)
fact_imdm_gold     = spark.table(fact_imdm_table).filter(F.col("ano_edicao") == reference_year)

checks = []
def check(controle, esperado, obtido):
    checks.append((controle, str(esperado), str(obtido), "PASS" if esperado == obtido else "FAIL"))

# ---- Esperados derivados da edição e do radar da edição ----
munic_count     = manifest_silver["linhas"]
recusa_contrato = manifest_silver["recusa_total"]

anchors_edicao = {anchor: (dim, regra) for dim, anchor, regra, _ in imdm_items}
esperado_ancoras  = len(anchors_edicao)
esperado_regras   = {}
esperado_peso_dim = {}
for cod, (dim, regra) in anchors_edicao.items():
    esperado_regras[regra]   = esperado_regras.get(regra, 0) + 1
    esperado_peso_dim[dim]   = esperado_peso_dim.get(dim, 0.0) + 1.0

# Municípios com Recusa formal em alguma variável de indicador da edição (da própria fact)
recusa_munis = (
    fact_resposta_gold.where(F.col("classe_preenchimento") == "Recusa")
    .select("cod_ibge").distinct().count()
)

# 1) Volume de indicadores
tema_counts = {r["tema"]: r["count"] for r in dim_indicador_gold.groupBy("tema").count().collect()}
temas_esperado = {
    "TIC": sum(1 for c in indicator_cols if c.startswith("mtic")),
    "GOV": sum(1 for c in indicator_cols if c.startswith("mgov")),
}
check(f"dim_indicador: {len(indicator_cols)} variáveis (colunas com dados da edição)", len(indicator_cols), dim_indicador_gold.count())

check(f"dim_indicador: temas TIC/GOV {temas_esperado}", temas_esperado, tema_counts
      )
# Radar da edição: âncoras com peso e distribuição por regra
ancoras_gold = dim_indicador_gold.where(F.col("peso_indicador").isNotNull())

check(f"dim_indicador: {esperado_ancoras} âncoras do radar presentes na edição", esperado_ancoras, ancoras_gold.count())
regra_counts = {r["regra_pontuacao"]: r["count"] for r in ancoras_gold.groupBy("regra_pontuacao").count().collect()}
check(
    f"dim_indicador: itens por regra {esperado_regras}",
    esperado_regras,
    regra_counts,
)
reconstruidas = ["mtic11", "mtic31", "mtic331", "mtic2421", "mtic2431", "mgov141", "mgov142"]
reconstruidas_edicao = [c for c in reconstruidas if c in indicator_cols]
check(
    f"Ajuste 2: significados preenchidos nas {len(reconstruidas_edicao)} reconstruídas presentes na edição",
    0,
    dim_indicador_gold.where(F.col("cod_variable").isin(reconstruidas_edicao) & F.col("meaning").isNull()).count(),
)
# Correção documental do grupo: mtic30 é COMUNICAÇÃO (o IBGE traz 'educação' por erro)
if "mtic30" in indicator_cols:
    mtic30_meaning = dim_indicador_gold.where(F.col("cod_variable") == "mtic30").select("meaning").collect()[0]["meaning"]
    check(
        "Ajuste 2: mtic30 corrigido para 'comunicação'",
        True,
        (mtic30_meaning is not None) and ("comunicação" in mtic30_meaning.lower()),
    )
else:
    print(f"mtic30 sem dados na edição {reference_year} — correção documental não aplicável (skip)")

# 2) Conservação exata de observações e das 6 classes da Etapa 2
classe_dist = {
    r["classe_preenchimento"]: r["count"]
    for r in fact_resposta_gold.groupBy("classe_preenchimento").count().collect()
}
esperado_linhas = munic_count * len(indicator_cols)
check(f"fact_resposta: {esperado_linhas} linhas ({munic_count} × {len(indicator_cols)})", esperado_linhas, fact_resposta_gold.count())
check(
    "fact_resposta: Recusa preservada (contrato da silver/edição)",
    recusa_contrato,
    classe_dist.get("Recusa", 0),
)
check(
    "fact_resposta: classes fecham o total (5 classes mutuamente exclusivas)",
    esperado_linhas,
    sum(classe_dist.values()),
)

# Denominador efetivo: o município "completo" tem peso = nº de itens da dimensão no radar da edição.
# Pega regra aplicada errada que os demais controles não enxergam (peso zerado ou duplicado).
peso_max_dim = {
    r["dimensao_imdm"]: r["peso_max"]
    for r in (
        fact_resposta_gold
        .join(dim_indicador_gold.select("cod_variable", "ano_edicao", "dimensao_imdm"), ["cod_variable", "ano_edicao"])
        .where(F.col("dimensao_imdm") != "Nao_Pontua")
        .groupBy("cod_ibge", "dimensao_imdm").agg(F.sum("peso_aplicado").alias("peso"))
        .groupBy("dimensao_imdm").agg(F.max("peso").alias("peso_max"))
        .collect()
    )
}
check(
    f"fact_resposta: peso máximo por dimensão = nº de itens da edição {esperado_peso_dim}",
    esperado_peso_dim,
    peso_max_dim,
)

# 3) IMDM: dimensões da edição, nulos explicados e recálculo independente
dims_imdm = [c for c, d in dims_edicao.items() if d["entra"]]
check(f"fact_score_dimensao: {munic_count} × {len(dims_edicao)} dimensões da edição",
      munic_count * len(dims_edicao), fact_score_gold.count())
sem_item_valido = (
    fact_score_gold.where(F.col("entra_no_imdm") & F.col("score").isNull())
    .select("cod_ibge").distinct()
    .join(fact_resposta_gold.where(F.col("classe_preenchimento") == "Recusa").select("cod_ibge").distinct(),
          "cod_ibge", "left_anti")
    .count()
)
null_imdm = fact_imdm_gold.where(F.col("score_global_imdm").isNull())
check(f"fact_imdm: {munic_count} municípios", munic_count, fact_imdm_gold.count())
check(f"fact_imdm: IMDM NULL = {recusa_munis} com Recusa + {sem_item_valido} sem item válido",
      recusa_munis + sem_item_valido, null_imdm.count())
check("fact_imdm: motivo 'Recusa' nos municípios com recusa", recusa_munis, null_imdm.where(F.col("motivo_imdm_nulo") == "Recusa").count())
valid_stats = fact_imdm_gold.where(F.col("score_global_imdm").isNotNull()).agg(
    F.count("*").alias("n"), F.min("score_global_imdm").alias("mn"), F.max("score_global_imdm").alias("mx")
).collect()[0]
check("fact_imdm: municípios válidos = total - NULL", munic_count - recusa_munis - sem_item_valido, valid_stats["n"])
check("fact_imdm: scores dentro de [0, 100]", True,
      valid_stats["n"] == 0 or ((valid_stats["mn"] >= 0) and (valid_stats["mx"] <= 100)))
recalc = (
    fact_score_gold.where(F.col("entra_no_imdm"))
    .groupBy("cod_ibge")
    .agg((F.sum(F.col("dimensao_peso") * F.col("score")) / F.sum("dimensao_peso")).alias("imdm_recalc"),
         F.count("score").alias("n"))
    .where(F.col("n") == len(dims_imdm))
    .join(fact_imdm_gold.select("cod_ibge", "score_global_imdm"), "cod_ibge")
    .where(F.col("score_global_imdm").isNotNull())
    .where(F.abs(F.col("imdm_recalc") - F.col("score_global_imdm")) > 0.011)
    .count()
)
check("fact_imdm: IMDM = média ponderada das dimensões (recálculo independente)", 0, recalc)

# 4) Integridade referencial (FK)
check(
    "FK: fact_resposta -> dim_municipio (0 órfãos)",
    0,
    fact_resposta_gold.join(dim_municipio_gold.select("cod_ibge", "ano_edicao"), ["cod_ibge", "ano_edicao"], "left_anti").count(),
)
check(
    "FK: fact_resposta -> dim_indicador (0 órfãos)",
    0,
    fact_resposta_gold.join(dim_indicador_gold.select("cod_variable", "ano_edicao"), ["cod_variable", "ano_edicao"], "left_anti").count(),
)
check(
    "FK: fact_score_dimensao -> dim_municipio (0 órfãos)",
    0,
    fact_score_gold.join(dim_municipio_gold.select("cod_ibge", "ano_edicao"), ["cod_ibge", "ano_edicao"], "left_anti").count(),
)
check(
    "FK: fact_imdm -> dim_municipio (0 órfãos)",
    0,
    fact_imdm_gold.join(dim_municipio_gold.select("cod_ibge", "ano_edicao"), ["cod_ibge", "ano_edicao"], "left_anti").count(),
)

# 5) Proteção ética/privacidade de servidores
eticas_edicao = [c for c in ("mgov15", "mgov16") if c in indicator_cols]
check(
    f"Ética: mgov15/mgov16 com regra NAO_PONTUA ({len(eticas_edicao)} presentes na edição)",
    len(eticas_edicao),
    dim_indicador_gold.where(
        F.col("cod_variable").isin(eticas_edicao) & (F.col("regra_pontuacao") == "NAO_PONTUA")
    ).count(),
)

# Seção 3.3.3: no 'não se aplica' legítimo de mgov05, pontos_obtidos também é NULL
if ancoras_excecao_lai:
    check(
        "Ajuste 3.3.3: não-aplica da LAI sem pontos (NULL)",
        0,
        fact_resposta_gold.where(
            F.col("cod_variable").isin(ancoras_excecao_lai)
            & (F.col("peso_aplicado").isNull())
            & (F.col("classe_preenchimento") == "Valida")
            & F.col("pontos_obtidos").isNotNull()
        ).count(),
    )

# Seção 3.2.1: grupo_pergunta nunca fica com string vazia — pai confirmado, radical regex ou NULL
check(
    "Ajuste 3.2.1: grupo_pergunta sem strings vazias",
    0,
    dim_indicador_gold.where(F.col("grupo_pergunta") == "").count(),
)

results = spark.createDataFrame(checks, schema="controle string, esperado string, obtido string, status string")
display(results)

failed = [c[0] for c in checks if c[3] == "FAIL"]
assert not failed, f"Controles de qualidade da camada Gold REPROVADOS: {failed}"
print("Todos os controles de qualidade da Etapa 3 (Seção 6) passaram.")

In [ ]:
# Declara o contrato da GOLD no ledger e fecha o invariante da cadeia: a recusa
# que nasceu na bronze, preservada pela silver, chega intacta à fact_resposta
# (docs/arquitetura/pipeline_manifest.md, seções 4 e 8).
n_pass = sum(1 for c in checks if c[3] == "PASS")

gravar_linha_manifest(
    etapa="gold",
    ano_edicao=reference_year,
    tabela_destino=f"{dim_municipio_table};{dim_indicador_table};{fact_resposta_table};{fact_score_dimensao_table};{fact_imdm_table}",
    linhas=fact_imdm_gold.count(),
    indicadores=int(dim_indicador_gold.count()),
    recusa_total=int(fact_resposta_gold.filter(F.col("classe_preenchimento") == "Recusa").count()),
    detalhes={
        "checks_pass": f"{n_pass}/{len(checks)}",
        "fact_resposta_linhas": str(fact_resposta_gold.count()),
        "fact_imdm_linhas": str(fact_imdm_gold.count()),
        "ancoras": str(ancoras_gold.count()),         # âncoras do radar com peso na edição (verificadas no Step 9)
        "radar_itens": str(esperado_ancoras),         # itens do radar da edição (config.radar_variavel)
        "radar_sha256": manifest_radar["sha256_xlsx"],
        "dimensoes_edicao": " | ".join(f"{d['ordem']}. {d['nome']}{'' if d['entra'] else ' (fora do IMDM)'}"
                                       for d in sorted(dims_edicao.values(), key=lambda d: d["ordem"])),
        "imdm_nulos": str(fact_imdm_gold.where(F.col("score_global_imdm").isNull()).count()),
    },
)